# Dependencies and Data

In [1]:
import pandas as pd
import geopandas as gpd
import linref as lr

In [2]:
LRS =   lr.LRS(key_col=['NLFID', 'chain'], loc_col='COUNTY_LOG_NBR', beg_col='CTL_BEGIN_', end_col='CTL_END_NB', geom_col='geometry', geom_m_col='geometry_m', closed='left_mod')
lr.set_default_lrs(LRS)

In [9]:
# Load linear events with geometry
roads =   gpd.read_file('testing/data/roadways.json').explode().rename(columns={'NLF_ID': 'NLFID'})
crashes = gpd.read_file('testing/data/crashes.json')

Skipping field INTERSECTION_LEG_ID: unsupported OGR type: 5


# Preparing LRS Data

In [15]:
# Add M values and chaining
roads = roads.lr.add_geom_m()
roads = roads.lr.add_chaining()

In [22]:
roads.lr.relate(crashes)

AttributeError: 'NoneType' object has no attribute 'dtype'

# Dissolving Linear Data

In [5]:
# Perform dissolve
dissolved = roads.lr.dissolve()

# Resegmenting Linear Data at Regular Intervals

In [6]:
# Resegment the dissolved data at 0.5 mile intervals
segments = dissolved.lr.resegment(length=0.5, fill='balance')

# Integrate Multiple Sets of Linear Events

In [7]:
# Create additional resegmented linear events for example
segments_1 = dissolved.lr.resegment(length=0.5, fill='balance').sample(frac=0.5, random_state=1)
segments_2 = dissolved.lr.resegment(length=0.3, fill='balance').sample(frac=0.5, random_state=1)
# Integrate the resegmented linear events
integrated = lr.integrate([segments_1, segments_2])
# Cut new geometry at integrated locations
integrated = integrated.lr.cut_from(dissolved)

# Project Point Events onto the LRS

In [8]:
# Project crash points onto the dissolved linear events
projected = dissolved.lr.project(crashes.drop(columns=['COUNTY_LOG_NBR', 'NLFID']))
# Interpolate new geometry at projected locations along roadway events
projected = projected.lr.interpolate_from(dissolved)